# Task 5: Semantic Similarity Search with Sentence Embeddings

**Goal:** Index a small text corpus and retrieve documents by semantic meaning using top-k search.

| Covers | Details |
|--------|---------|
| Encoding | `sentence-transformers` — `all-MiniLM-L6-v2` (384-dim) |
| Indexing  | FAISS `IndexFlatIP` (exact cosine similarity) |
| Retrieval | Top-k nearest-neighbor search |
| Visualization | t-SNE embedding space plot |
| Bonus | Pinecone integration pattern |

**References**
- [Sentence Embeddings Introduction (video)](https://youtu.be/6BEMXQBIysg)
- [Pinecone Vector DB Intro (video)](https://youtu.be/7gF9l3MI-0E)
- [OpenAI Embeddings Concepts](https://platform.openai.com/docs/guides/embeddings)
- [FAISS Official Docs](https://github.com/facebookresearch/faiss)
- [Pinecone Docs](https://docs.pinecone.io)


## 1. Core Concepts

### What is a sentence embedding?
A sentence embedding maps raw text to a fixed-length dense vector. Semantically similar sentences produce vectors that point in the same direction in high-dimensional space. `all-MiniLM-L6-v2` outputs 384-dimensional vectors and runs fast on CPU.

### What is FAISS?
FAISS (Facebook AI Similarity Search) builds an index over those vectors and returns the **k** nearest neighbors to a query vector — even at billions of vectors — in milliseconds. No infrastructure required; runs entirely in-process.

### Cosine similarity
Measures the angle between two vectors. Score of `1.0` = identical direction (same meaning). Score of `0.0` = orthogonal (unrelated). After L2-normalization, inner product equals cosine similarity, so FAISS `IndexFlatIP` gives exact cosine scores.

### Full pipeline
```
raw text
   ↓  sentence encoder
384-dim L2-normalized vector
   ↓  FAISS IndexFlatIP
k nearest neighbors (score, doc_id)
   ↓  metadata lookup
top-k results with original text
```


## 2. Setup

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import faiss
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE

print("Imports OK.")
print(f"FAISS version : {faiss.__version__}")


## 3. The Corpus

20 short documents across **4 topics**: AI/ML, vector databases, finance/fraud, and sports.

The topic labels are ground truth. A working retrieval system should surface same-topic documents for same-topic queries — this is the test.


In [ ]:
# ── 20-document corpus ────────────────────────────────────────────────────────
CORPUS = [
    # ── AI / Machine Learning (docs 0-4) ─────────────────────────────────────
    "Neural networks learn by adjusting weights through backpropagation to minimize prediction error.",
    "Transformers rely on self-attention to model long-range dependencies in text without recurrence.",
    "Transfer learning lets a pretrained model adapt to new tasks with far less labeled data.",
    "Gradient descent iteratively updates model parameters in the direction that reduces the loss.",
    "Overfitting occurs when a model memorizes training data instead of learning generalizable patterns.",

    # ── Vector Databases & Semantic Search (docs 5-9) ────────────────────────
    "FAISS indexes high-dimensional vectors for millisecond-scale approximate nearest neighbor search.",
    "Pinecone is a managed vector database that removes infrastructure overhead from AI similarity search.",
    "Cosine similarity measures the angle between two vectors, returning 1.0 for identical directions.",
    "ANN algorithms like HNSW trade a small accuracy loss for orders-of-magnitude speed gains at scale.",
    "Semantic search retrieves documents by meaning, not keyword overlap, using dense vector representations.",

    # ── Finance & Fraud Detection (docs 10-14) ───────────────────────────────
    "Invoice fraud involves submitting fictitious or inflated bills to siphon funds from an organization.",
    "Anomaly detection flags transactions that deviate significantly from a user's normal spending patterns.",
    "Bid rigging occurs when suppliers secretly coordinate to fix prices and divide contract awards.",
    "KYC procedures verify customer identity to prevent money laundering and terrorist financing.",
    "Forensic accounting traces financial records to uncover asset misappropriation and hidden losses.",

    # ── Sports & Athletics (docs 15-19) ──────────────────────────────────────
    "A midfielder who reads the press triggers a counterattack before the defense resets.",
    "The pick-and-roll is a two-player basketball action that creates mismatches and open shots.",
    "Marathon training builds aerobic base through high weekly mileage and long slow distance runs.",
    "A fast bowler in cricket varies pace and seam position to beat the batsman's footwork.",
    "Sprint mechanics training focuses on hip extension, ground contact time, and forward lean.",
]

LABELS  = ["ai_ml"] * 5 + ["vector_db"] * 5 + ["finance_fraud"] * 5 + ["sports"] * 5
DOC_IDS = [f"doc_{i:02d}" for i in range(len(CORPUS))]

df = pd.DataFrame({"id": DOC_IDS, "topic": LABELS, "text": CORPUS})

print(f"Corpus size : {len(CORPUS)} documents")
print(f"Topic counts: {df['topic'].value_counts().to_dict()}")
df.head(8)


## 4. Encoding Documents

`all-MiniLM-L6-v2` is a lightweight sentence transformer (22 M params, 384-dim) fine-tuned on semantic similarity tasks. It runs fast on CPU and is the standard starting point for semantic search.

`normalize_embeddings=True` applies L2 normalization. After normalization, inner product equals cosine similarity, which is what `IndexFlatIP` computes.


In [ ]:
# ── Load model + encode corpus ────────────────────────────────────────────────
MODEL_NAME = "all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

print(f"Model              : {MODEL_NAME}")
print(f"Max sequence length: {model.max_seq_length} tokens")
print("Encoding corpus...")

embeddings = model.encode(
    CORPUS,
    show_progress_bar=True,
    normalize_embeddings=True,   # L2 norm → dot product = cosine similarity
    convert_to_numpy=True,
)

print(f"\nEmbedding matrix : {embeddings.shape}  ({embeddings.shape[0]} docs × {embeddings.shape[1]} dims)")
print(f"L2 norm of doc[0]: {np.linalg.norm(embeddings[0]):.6f}  (should be 1.000000)")
print(f"dtype            : {embeddings.dtype}")


## 5. Building the FAISS Index

`IndexFlatIP` stores all vectors flat (no compression) and computes exact inner products at query time. No training step, fully deterministic results. The right choice for datasets up to ~100k vectors.

For larger scale:
- `IndexIVFFlat` — clusters vectors into Voronoi cells, searches only nearby cells
- `IndexIVFPQ` — adds product quantization to compress vectors and cut memory
- Pinecone / Weaviate / Qdrant — managed services that handle this automatically


In [ ]:
# ── Build FAISS index ─────────────────────────────────────────────────────────
DIM = embeddings.shape[1]   # 384 for all-MiniLM-L6-v2

index = faiss.IndexFlatIP(DIM)          # exact cosine similarity (normalized vectors)
index.add(embeddings.astype("float32")) # add all 20 document vectors

print(f"Index type      : IndexFlatIP (exact inner product)")
print(f"Vector dimension: {DIM}")
print(f"Vectors in index: {index.ntotal}")
print(f"Is trained      : {index.is_trained}  (FlatIndex needs no training)")


## 6. Semantic Retrieval

`retrieve()` encodes the query with the same model (preserving the same vector space), then calls `index.search()` for the top-k nearest vectors.

Cosine scores: `0.7+` = very strong match, `0.5–0.7` = clear match, `< 0.4` = weak/unrelated.


In [ ]:
# ── Query function ────────────────────────────────────────────────────────────
def retrieve(query: str, k: int = 5) -> pd.DataFrame:
    """Return top-k documents semantically closest to query.

    Args:
        query: free-text search string
        k:     number of results to return

    Returns:
        DataFrame with columns [rank, score, topic, doc_id, text]
    """
    query_vec = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(query_vec, k)

    rows = []
    for rank, (doc_id, score) in enumerate(zip(ids[0], scores[0]), start=1):
        rows.append({
            "rank"  : rank,
            "score" : round(float(score), 4),
            "topic" : LABELS[doc_id],
            "doc_id": DOC_IDS[doc_id],
            "text"  : CORPUS[doc_id],
        })
    return pd.DataFrame(rows)


# ── Single query demo ─────────────────────────────────────────────────────────
DEMO_QUERY = "How do neural networks learn from training data?"

print(f'Query: "{DEMO_QUERY}"')
print("=" * 72)

results = retrieve(DEMO_QUERY, k=5)
for _, row in results.iterrows():
    print(f"  [{row['rank']}] score={row['score']:.4f}  topic={row['topic']}")
    print(f"      {row['text']}")
    print()


## 7. Multi-Query Demo

Five queries — one per topic plus one cross-topic query. Each should surface same-topic documents at the top.

The cross-topic query "How do you find the most similar item in a large dataset?" contains no vector-DB keywords but should still land in `vector_db` because the meaning maps there.


In [ ]:
# ── Five queries across all topics ───────────────────────────────────────────
DEMO_QUERIES = [
    ("AI/ML",         "What optimization algorithm trains deep learning models?"),
    ("Vector DB",     "How does similarity search work in a vector database?"),
    ("Finance/Fraud", "What techniques detect financial corruption in procurement?"),
    ("Sports",        "How do athletes build endurance for long-distance events?"),
    ("Cross-topic",   "How do you find the most similar item in a large dataset?"),  # → vector_db
]

TOP_K = 3

for label, q in DEMO_QUERIES:
    r = retrieve(q, k=TOP_K)
    print(f"\n{'=' * 72}")
    print(f"  [{label}]")
    print(f"  Q: {q}")
    print(f"{'─' * 72}")
    for _, row in r.iterrows():
        tag = "✓" if row["topic"] in ("vector_db" if label == "Cross-topic" else label.lower().replace("/", "_").replace("-", "_")) else " "
        print(f"  {row['rank']}. score={row['score']:.4f}  [{row['topic']}]  {row['text'][:65]}...")

print("\n✓ Retrieval demo complete.")


## 8. Embedding Space Visualization

t-SNE reduces 384-dimensional vectors to 2D for plotting. Documents in the same semantic cluster should appear close together. This is visual confirmation that the encoder captures topic structure before any indexing happens.


In [ ]:
# ── t-SNE reduction + scatter plot ───────────────────────────────────────────
TOPIC_COLORS = {
    "ai_ml"         : "#4C72B0",
    "vector_db"     : "#DD8452",
    "finance_fraud" : "#55A868",
    "sports"        : "#C44E52",
}

tsne = TSNE(n_components=2, perplexity=5, random_state=42, init="pca", max_iter=1000)
reduced = tsne.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(9, 7))
ax.set_facecolor("#12122a")
fig.patch.set_facecolor("#12122a")

for topic, color in TOPIC_COLORS.items():
    mask = np.array([l == topic for l in LABELS])
    ax.scatter(
        reduced[mask, 0], reduced[mask, 1],
        color=color, label=topic,
        s=150, edgecolors="white", linewidths=0.6, zorder=3,
    )

ax.legend(
    title="Topic", loc="upper left",
    facecolor="#1e1e3f", edgecolor="#555",
    labelcolor="white", title_fontsize=9, fontsize=9,
)
ax.set_title("Sentence Embedding Space  (t-SNE: 384D → 2D)", color="white", fontsize=13, pad=14)
ax.set_xlabel("t-SNE dim 1", color="#aaa")
ax.set_ylabel("t-SNE dim 2", color="#aaa")
ax.tick_params(colors="#666")
for spine in ax.spines.values():
    spine.set_edgecolor("#333")

plt.tight_layout()
plt.savefig("embedding_tsne.png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print("Saved → embedding_tsne.png")


## 9. Pinecone: Managed Vector Database

FAISS runs in-process with no persistence. Pinecone handles infrastructure: persistent storage, distributed search, metadata filtering, and real-time updates. Same retrieval logic, different plumbing.

**Three Pinecone primitives:**
1. `pc.create_index(...)` — define dimension + similarity metric
2. `index.upsert(vectors)` — store `{id, values, metadata}` records
3. `index.query(...)` — top-k search with optional metadata filter

Get a free API key at https://app.pinecone.io — the starter tier handles this dataset with room to spare.


In [ ]:
# ── Pinecone integration pattern ─────────────────────────────────────────────
# pip install pinecone-client
# Requires: PINECONE_API_KEY environment variable

# from pinecone import Pinecone, ServerlessSpec
# import os

# pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# ── 1. Create index ───────────────────────────────────────────────────────────
# INDEX_NAME = "task5-demo"
# existing = [i.name for i in pc.list_indexes().indexes]
# if INDEX_NAME not in existing:
#     pc.create_index(
#         name=INDEX_NAME,
#         dimension=384,        # must match the model output dimension
#         metric="cosine",
#         spec=ServerlessSpec(cloud="aws", region="us-east-1"),
#     )
# pin_index = pc.Index(INDEX_NAME)

# ── 2. Upsert vectors ─────────────────────────────────────────────────────────
# vectors = [
#     {
#         "id"      : DOC_IDS[i],
#         "values"  : embeddings[i].tolist(),
#         "metadata": {"text": CORPUS[i], "topic": LABELS[i]},
#     }
#     for i in range(len(CORPUS))
# ]
# pin_index.upsert(vectors=vectors)
# print(pin_index.describe_index_stats())

# ── 3. Query ──────────────────────────────────────────────────────────────────
# def pinecone_retrieve(query: str, k: int = 5, topic_filter: str = None):
#     qvec = model.encode([query], normalize_embeddings=True).tolist()[0]
#     filter_expr = {"topic": {"$eq": topic_filter}} if topic_filter else None
#     result = pin_index.query(vector=qvec, top_k=k, include_metadata=True, filter=filter_expr)
#     for m in result["matches"]:
#         print(f"  score={m['score']:.4f}  [{m['metadata']['topic']}]  {m['metadata']['text']}")

# pinecone_retrieve("What optimization algorithm trains deep learning models?")
# pinecone_retrieve("fraud detection", k=3, topic_filter="finance_fraud")  # with metadata filter

print("Pinecone pattern above — uncomment and add your API key to run.")
print("Structurally identical to FAISS: encode query → nearest neighbors → metadata lookup.")


## 10. Summary

| Step | What happened |
|------|--------------|
| **Encode** | `all-MiniLM-L6-v2` mapped each sentence to a 384-dim L2-normalized vector |
| **Index** | FAISS `IndexFlatIP` stored 20 vectors for exact cosine-similarity lookup |
| **Query** | Encoding the query and calling `index.search()` returned top-k results in < 1ms |
| **Verify** | Same-topic queries consistently surfaced same-topic documents |
| **Visualize** | t-SNE showed 4 distinct topic clusters in the embedding space |

### Key insight
Keyword overlap is irrelevant. The query "What optimization algorithm trains deep learning models?" surfaces gradient descent and backpropagation even though neither phrase appears in the query — because the embedding captures meaning, not surface form.

### Production upgrade path

| Scale | Switch to |
|-------|-----------|
| > 100k docs | `IndexIVFPQ` (approximate, compressed) |
| Persistence required | Pinecone / Qdrant / Weaviate |
| Higher accuracy | `all-mpnet-base-v2` (768-dim) |
| Domain-specific | Fine-tune on your own query-passage pairs |

---
*Task 5 / 30 — AI Engineering curriculum — Annalyxx Global Ltd*
